# LocalStoryChat: Qwen3.5-9B Writer QLoRA on Kaggle

First writer-training experiment using a Kaggle T4x2 notebook. The notebook clones the `generic-story-engine-rebuild` branch, rebuilds and validates the 48-example Gold dataset, preprocesses the training config, then runs the conservative T4-compatible QLoRA job.

In [ ]:
import subprocess, sys
print(sys.version)
subprocess.run(['nvidia-smi'], check=False)


## 1. Kaggle settings

In the Kaggle notebook Settings panel, select **Accelerator = T4x2** and enable **Internet**. The run uses GPU 0 for the first experiment so the setup stays simple.

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/LocalStoryChat'):
    subprocess.run(['git','clone','--branch','generic-story-engine-rebuild','--single-branch','https://github.com/tony21675/LocalStoryChat.git','/kaggle/working/LocalStoryChat'], check=True)
os.chdir('/kaggle/working/LocalStoryChat')
print(os.getcwd())


## 2. Install Axolotl

This uses the current Axolotl release line that supports Qwen3.5.

In [ ]:
%pip install -q --no-build-isolation "axolotl>=0.16.1"


If Kaggle reports that packages changed underneath the running kernel, use **Run → Restart Session**, then continue with the next cells.

In [ ]:
import os, subprocess
os.chdir('/kaggle/working/LocalStoryChat')
subprocess.run(['python','training/prepare_gold_dataset.py'], check=True)


In [ ]:
from pathlib import Path
gold = sorted(Path('training/gold').glob('*.json'))
dataset = Path('training/gold_dataset.jsonl')
print('Gold examples:', len(gold))
print('Dataset exists:', dataset.exists())
if dataset.exists():
    print('Dataset size:', dataset.stat().st_size, 'bytes')


## 3. Preprocess before training

This step is intentionally separate from training so a bad config, chat template, or dataset format fails before we spend GPU time.

In [ ]:
import os, subprocess
os.chdir('/kaggle/working/LocalStoryChat')
subprocess.run(['axolotl','preprocess','training/qwen3_5_9b_kaggle_t4.yaml'], check=True)


## 4. Train the writer adapter

Settings: 4-bit QLoRA, LoRA rank 16, sequence length 2048, micro-batch 1, gradient accumulation 8, 3 epochs, FP16, SDPA, no sample packing.

In [ ]:
import os, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.chdir('/kaggle/working/LocalStoryChat')
subprocess.run(['axolotl','train','training/qwen3_5_9b_kaggle_t4.yaml'], check=True)


## 5. Inspect and package the adapter

In [ ]:
from pathlib import Path
out = Path('training/outputs/qwen3_5_9b_writer_kaggle')
print('Output exists:', out.exists())
if out.exists():
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(p)


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/qwen3_5_9b_writer_kaggle_adapter', 'zip', 'training/outputs/qwen3_5_9b_writer_kaggle')
print(archive)
